# Lab 3


[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/giswqs/geog-414/blob/master/book/labs/lab_03.ipynb)

## Submission requirements

1. Upload a screenshot of your map for each question.
2. Provide a link to your notebook on Colab. See instructions [here](https://geog-414.gishub.org/book/labs/instructions.html).


## Question 1

Find a DEM dataset in the [Earth Engine Data Catalog](https://developers.google.com/earth-engine/datasets/), clip the DEM to your home state, and display the DEM with a proper color palette on the map. For example, the sample map below shows the DEM of the state of Colorado.

In [3]:
import ee
import geemap

PROJECT_ID = 'lab3-515123'

# احراز هویت و مقداردهی اولیه Earth Engine
ee.Authenticate()
ee.Initialize(project=PROJECT_ID)

# ساخت نقشه تعاملی و تنظیم نقشه پایه CartoDB.Positron
m = geemap.Map(center=[39, -105.5], zoom=7, height='600px')
m.clear_layers()
m.add_basemap('CartoDB.Positron')

# بارگذاری مرز ایالت کلرادو
states = ee.FeatureCollection('TIGER/2018/States')
colorado = states.filter(ee.Filter.eq('NAME', 'Colorado'))

# بارگذاری داده ارتفاعی NASADEM و برش روی کلرادو
dem = ee.Image('NASA/NASADEM_HGT/001').select('elevation')
dem_clipped = dem.clip(colorado)

# تنظیمات نمایش و پالت رنگی
dem_vis = {
    'min': 0.0,
    'max': 4000.0,
    'palette': ['0000ff', '00ffff', '00ff00', 'ffff00', 'ff0000', 'ffffff']
}

# افزودن لایه‌ها به نقشه
m.addLayer(dem_clipped, dem_vis, 'DEM Colorado')
m.addLayer(colorado.style(color='red', fillColor='00000000'), {}, 'Colorado Border')

# افزودن متن نام و لوگو
m.add_text('Made by Mahdieh Zakizadeh', fontsize=20, position='bottomright')
m.add_image('https://i.imgur.com/at4Qprk.png', position='bottomright')

# نمایش نقشه
m

Basemap can only be one of the following:
  OpenStreetMap
  Esri.WorldStreetMap
  Esri.WorldImagery
  Esri.WorldTopoMap
  FWS NWI Wetlands
  FWS NWI Wetlands Raster
  NLCD 2021 CONUS Land Cover
  NLCD 2019 CONUS Land Cover
  NLCD 2016 CONUS Land Cover
  NLCD 2013 CONUS Land Cover
  NLCD 2011 CONUS Land Cover
  NLCD 2008 CONUS Land Cover
  NLCD 2006 CONUS Land Cover
  NLCD 2004 CONUS Land Cover
  NLCD 2001 CONUS Land Cover
  USGS NAIP Imagery
  USGS NAIP Imagery False Color
  USGS NAIP Imagery NDVI
  USGS Hydrography
  USGS 3DEP Elevation
  ESA Worldcover 2020
  ESA Worldcover 2020 S2 FCC
  ESA Worldcover 2020 S2 TCC
  ESA Worldcover 2021
  ESA Worldcover 2021 S2 FCC
  ESA Worldcover 2021 S2 TCC
  BaseMapDE.Color
  BaseMapDE.Grey
  BasemapAT.basemap
  BasemapAT.grau
  BasemapAT.highdpi
  BasemapAT.orthofoto
  BasemapAT.overlay
  BasemapAT.surface
  BasemapAT.terrain
  CyclOSM
  Esri.AntarcticBasemap
  Esri.AntarcticImagery
  Esri.ArcticImagery
  Esri.ArcticOceanBase
  Esri.ArcticOceanRe

Map(center=[39, -105.5], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright', …

![](https://i.imgur.com/OLeSt7n.png)

## Question 2

Use [Sentinel-2](https://developers.google.com/earth-engine/datasets/catalog/sentinel-2) or [Landsat-9 data](https://developers.google.com/earth-engine/datasets/catalog/landsat-9), create a cloud-free imagery of the year 2021 for your home state. Display the imagery on the map with a proper band combination. For example, the sample map below shows a cloud-free false-color composite of Sentinel-2 imagery of the year 2021 for the state of Colorado.

In [4]:
# Add your code here.import ee
import geemap

PROJECT_ID = 'lab3-515123'

# مقداردهی اولیه Earth Engine (در صورت نیاز ابتدا ee.Authenticate() را اجرا کنید)
ee.Initialize(project=PROJECT_ID)

# ساخت نقشه تعاملی
m = geemap.Map(center=[39, -105.5], zoom=7, height='600px')
m.clear_layers()
m.add_basemap('CartoDB.Positron')

# مرز ایالت کلرادو
states = ee.FeatureCollection('TIGER/2018/States')
colorado = states.filter(ee.Filter.eq('NAME', 'Colorado'))


# تابع نقاب‌گذاری ابر و سایه ابر با استفاده از باند SCL در Sentinel-2 SR
def mask_s2_clouds(image):
  scl = image.select('SCL')
  # حذف پیکسل‌های ابر متوسط (3)، ابر بالا (8)، ابر سیروس (9) و سایه ابر (10)
  clear_mask = (
      scl.neq(3).And(scl.neq(8)).And(scl.neq(9)).And(scl.neq(10))
  )
  return image.updateMask(clear_mask)


# بارگذاری، فیلتر زمانی (سال 2021)، حذف ابر و ساخت تصویر مرکب بدون ابر (Median)
s2_2021 = (
    ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
    .filterBounds(colorado)
    .filterDate('2021-01-01', '2021-12-31')
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 30))
    .map(mask_s2_clouds)
    .median()
    .clip(colorado)
)

# تنظیمات نمایش ترکیب رنگی کاذب (False Color: B8, B4, B3)
vis_params = {'bands': ['B8', 'B4', 'B3'], 'min': 200, 'max': 4000, 'gamma': 1.1}

# افزودن لایه‌ها به نقشه
m.addLayer(s2_2021, vis_params, 'Sentinel-2 False Color 2021 (Colorado)')
m.addLayer(
    colorado.style(color='yellow', fillColor='00000000', width=2),
    {},
    'Colorado Border',
)

# افزودن متن نام و لوگو
m.add_text('Made by Mahdieh Zakizadeh', fontsize=20, position='bottomright')
m.add_image('https://i.imgur.com/at4Qprk.png', position='bottomright')

# نمایش نقشه
m

Basemap can only be one of the following:
  OpenStreetMap
  Esri.WorldStreetMap
  Esri.WorldImagery
  Esri.WorldTopoMap
  FWS NWI Wetlands
  FWS NWI Wetlands Raster
  NLCD 2021 CONUS Land Cover
  NLCD 2019 CONUS Land Cover
  NLCD 2016 CONUS Land Cover
  NLCD 2013 CONUS Land Cover
  NLCD 2011 CONUS Land Cover
  NLCD 2008 CONUS Land Cover
  NLCD 2006 CONUS Land Cover
  NLCD 2004 CONUS Land Cover
  NLCD 2001 CONUS Land Cover
  USGS NAIP Imagery
  USGS NAIP Imagery False Color
  USGS NAIP Imagery NDVI
  USGS Hydrography
  USGS 3DEP Elevation
  ESA Worldcover 2020
  ESA Worldcover 2020 S2 FCC
  ESA Worldcover 2020 S2 TCC
  ESA Worldcover 2021
  ESA Worldcover 2021 S2 FCC
  ESA Worldcover 2021 S2 TCC
  BaseMapDE.Color
  BaseMapDE.Grey
  BasemapAT.basemap
  BasemapAT.grau
  BasemapAT.highdpi
  BasemapAT.orthofoto
  BasemapAT.overlay
  BasemapAT.surface
  BasemapAT.terrain
  CyclOSM
  Esri.AntarcticBasemap
  Esri.AntarcticImagery
  Esri.ArcticImagery
  Esri.ArcticOceanBase
  Esri.ArcticOceanRe

Map(center=[39, -105.5], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright', …

![](https://i.imgur.com/wD0VamV.png)